In [153]:
import pandas as pd
import numpy as np

In [154]:
folderPath = r"C:\Users\Aaron\OneDrive\Desktop\Capstone\Data\Features"

precipitation_raw = pd.read_csv(folderPath + r"\Weather\Precipitation1985_2025.csv")
sunshine_raw = pd.read_csv(folderPath + r"\Weather\SunshineDuration1985_2025.csv")
max_temp_raw = pd.read_csv(folderPath + r"\Weather\MaxTemp1985_2025.csv")
min_temp_raw = pd.read_csv(folderPath + r"\Weather\MinTemp1985_2025.csv")
meanTemp = pd.read_csv(folderPath + r"\Weather\AirTemperature1985_2025.csv")

precipitation_monthly = precipitation_raw.groupby(["Year", "Month"])["Measurement"].mean().reset_index()
sunshine_monthly = sunshine_raw.groupby(["Year", "Month"])["Measurement"].mean().reset_index()
maxTemp_monthly = max_temp_raw.groupby(["Year", "Month"])["Measurement"].mean().reset_index()
minTemp_monthly = min_temp_raw.groupby(["Year", "Month"])["Measurement"].mean().reset_index()

In [155]:
#Yield detrend = Actual tonnes -  Trend predicted tonnes
def detrend_yield(yield_data):

    yield_copy = yield_data.copy()

    x = yield_copy['Year']
    y = yield_copy['Tonnes']

    slope, intercept = np.polyfit(x,y, deg=1)

    #Predicted yield trend 
    yield_copy['Trend'] = slope * yield_copy['Year'] + intercept

    #Actual yield - predicted yield
    yield_copy['Yield_Detrended'] = yield_copy['Tonnes'] - yield_copy['Trend']

    return yield_copy

In [156]:
precip_m, sun_m, tmax_m, tmin_m = precipitation_monthly, sunshine_monthly, maxTemp_monthly, minTemp_monthly

def monthly_features(monthly_avg, colname):
        
        pivot = monthly_avg.pivot(index='Year', columns='Month', values='Measurement')
        pivot.columns = [f'{colname}_{m}' for m in pivot.columns]

        return pivot.reset_index()

In [157]:
features = (monthly_features(precip_m, 'Precip').merge(monthly_features(sun_m, 'Sun'), on='Year').merge(monthly_features(tmax_m, 'TempMax'), on='Year').merge(monthly_features(tmin_m, 'TempMin'), on='Year'))

features.head()

,Year,Precip_April,Precip_August,Precip_December,Precip_February,Precip_January,Precip_July,Precip_June,Precip_March,Precip_May,...,TempMin_December,TempMin_February,TempMin_January,TempMin_July,TempMin_June,TempMin_March,TempMin_May,TempMin_November,TempMin_October,TempMin_September
0,1985,54.266667,161.200000,114.566667,69.666667,67.533333,84.133333,84.800000,92.933333,65.066667,...,4.600000,3.300000,-0.233333,11.400000,8.466667,2.633333,6.900000,1.800000,8.333333,11.266667
1,1986,48.466667,166.433333,149.866667,5.533333,115.433333,55.600000,82.800000,103.000000,103.833333,...,3.166667,-0.166667,2.000000,11.733333,9.966667,2.533333,6.933333,4.700000,7.500000,7.666667
2,1987,74.766667,71.066667,93.600000,72.133333,51.200000,38.766667,114.733333,85.166667,22.366667,...,5.633333,2.266667,1.966667,12.066667,9.133333,2.400000,6.333333,4.833333,6.066667,9.266667
3,1988,41.666667,94.133333,59.000000,70.100000,182.633333,111.833333,47.633333,103.800000,84.100000,...,6.400000,2.300000,3.033333,10.666667,10.700000,4.400000,7.066667,4.666667,7.733333,9.700000
4,1989,66.066667,84.266667,124.200000,85.000000,70.633333,12.800000,62.400000,115.966667,15.100000,...,3.633333,3.566667,4.900000,13.233333,9.566667,3.600000,8.066667,5.500000,9.033333,10.166667


In [158]:
springWheat_yield = pd.read_csv(folderPath + r"\CropYield\SpringWheat1985_2025.csv")[['Year', 'Tonnes']]
yield_df = detrend_yield(springWheat_yield)

yield_df.head()

,Year,Tonnes,Trend,Yield_Detrended
0,1985,5.8,6.610453,-0.810453
1,1986,5.4,6.648101,-1.248101
2,1987,6.2,6.685749,-0.485749
3,1988,7.0,6.723397,0.276603
4,1989,6.2,6.761045,-0.561045


In [159]:
data = features.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)

data.head()

,Year,Precip_April,Precip_August,Precip_December,Precip_February,Precip_January,Precip_July,Precip_June,Precip_March,Precip_May,...,TempMin_July,TempMin_June,TempMin_March,TempMin_May,TempMin_November,TempMin_October,TempMin_September,Tonnes,Trend,Yield_Detrended
0,1985,54.266667,161.200000,114.566667,69.666667,67.533333,84.133333,84.800000,92.933333,65.066667,...,11.400000,8.466667,2.633333,6.900000,1.800000,8.333333,11.266667,5.8,6.610453,-0.810453
1,1986,48.466667,166.433333,149.866667,5.533333,115.433333,55.600000,82.800000,103.000000,103.833333,...,11.733333,9.966667,2.533333,6.933333,4.700000,7.500000,7.666667,5.4,6.648101,-1.248101
2,1987,74.766667,71.066667,93.600000,72.133333,51.200000,38.766667,114.733333,85.166667,22.366667,...,12.066667,9.133333,2.400000,6.333333,4.833333,6.066667,9.266667,6.2,6.685749,-0.485749
3,1988,41.666667,94.133333,59.000000,70.100000,182.633333,111.833333,47.633333,103.800000,84.100000,...,10.666667,10.700000,4.400000,7.066667,4.666667,7.733333,9.700000,7.0,6.723397,0.276603
4,1989,66.066667,84.266667,124.200000,85.000000,70.633333,12.800000,62.400000,115.966667,15.100000,...,13.233333,9.566667,3.600000,8.066667,5.500000,9.033333,10.166667,6.2,6.761045,-0.561045


In [160]:
X = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
y = data['Yield_Detrended']

In [161]:
X.head()

,Precip_April,Precip_August,Precip_December,Precip_February,Precip_January,Precip_July,Precip_June,Precip_March,Precip_May,Precip_November,...,TempMin_December,TempMin_February,TempMin_January,TempMin_July,TempMin_June,TempMin_March,TempMin_May,TempMin_November,TempMin_October,TempMin_September
0,54.266667,161.200000,114.566667,69.666667,67.533333,84.133333,84.800000,92.933333,65.066667,59.833333,...,4.600000,3.300000,-0.233333,11.400000,8.466667,2.633333,6.900000,1.800000,8.333333,11.266667
1,48.466667,166.433333,149.866667,5.533333,115.433333,55.600000,82.800000,103.000000,103.833333,119.466667,...,3.166667,-0.166667,2.000000,11.733333,9.966667,2.533333,6.933333,4.700000,7.500000,7.666667
2,74.766667,71.066667,93.600000,72.133333,51.200000,38.766667,114.733333,85.166667,22.366667,63.933333,...,5.633333,2.266667,1.966667,12.066667,9.133333,2.400000,6.333333,4.833333,6.066667,9.266667
3,41.666667,94.133333,59.000000,70.100000,182.633333,111.833333,47.633333,103.800000,84.100000,42.266667,...,6.400000,2.300000,3.033333,10.666667,10.700000,4.400000,7.066667,4.666667,7.733333,9.700000
4,66.066667,84.266667,124.200000,85.000000,70.633333,12.800000,62.400000,115.966667,15.100000,55.766667,...,3.633333,3.566667,4.900000,13.233333,9.566667,3.600000,8.066667,5.500000,9.033333,10.166667


In [162]:
y.head()

0   -0.810453
1   -1.248101
2   -0.485749
3    0.276603
4   -0.561045
Name: Yield_Detrended, dtype: float64

In [163]:
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()

In [164]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

def run_xgboost(X, y, params):

    y_true, y_pred = [], []

    for train_index, test_index in loo.split(X):

        model = xgb.XGBRegressor(random_state=42, **params)
        model.fit(X.iloc[train_index], y.iloc[train_index])
        
        y_pred.append(model.predict(X.iloc[test_index])[0])
        y_true.append(y.iloc[test_index].values[0])

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return rmse, r2

In [165]:
params_xgboost = {'max_depth': 1, 'n_estimators': 30, 'reg_alpha': 0.5, 'reg_lambda': 0.5}

In [166]:
crops_info = [
    (folderPath + r"\CropYield\SpringWheat1985_2025.csv", 'SpringWheat'),
    (folderPath + r"\CropYield\WinterWheat1985_2025.csv", 'WinterWheat'),
    (folderPath + r"\CropYield\SpringBarley1985_2025.csv", 'SpringBarley'),
    (folderPath + r"\CropYield\WinterBarley1985_2025.csv", 'WinterBarley'),
    (folderPath + r"\CropYield\SpringOats1985_2025.csv", 'SpringOats'),
    (folderPath + r"\CropYield\WinterOats1985_2025.csv", 'WinterOats'),
    (folderPath + r"\CropYield\Potatoes1985_2025.csv", 'Potatoes'),
    (folderPath + r"\CropYield\OilseedRape1985_2025.csv", 'OilseedRape'),
    (folderPath + r"\CropYield\BeansAndPeas1985_2025.csv", 'BeansAndPeas'),
]

### Testing All Month

In [167]:
print("All Months")

for csv_path, crop_name in crops_info:
    yield_df_crop = detrend_yield(pd.read_csv(csv_path)[['Year', 'Tonnes']])

    data_monthly = features.merge(yield_df_crop, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X_monthly = data_monthly.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y_monthly = data_monthly['Yield_Detrended']

    rmse, r2 = run_xgboost(X_monthly, y_monthly, params_xgboost)

    print(f"  {crop_name:15s} Features: {X_monthly.shape[1]}  RMSE: {rmse:.3f}  R2: {r2:.3f}")

All Months
  SpringWheat     Features: 48  RMSE: 0.846  R2: -0.584
  WinterWheat     Features: 48  RMSE: 0.799  R2: 0.187
  SpringBarley    Features: 48  RMSE: 0.641  R2: -0.224
  WinterBarley    Features: 48  RMSE: 0.891  R2: -0.702
  SpringOats      Features: 48  RMSE: 0.510  R2: -0.056
  WinterOats      Features: 48  RMSE: 0.502  R2: -0.159
  Potatoes        Features: 48  RMSE: 3.732  R2: -0.001
  OilseedRape     Features: 48  RMSE: 0.428  R2: 0.052
  BeansAndPeas    Features: 48  RMSE: 0.712  R2: -0.193


### Testing Critical Months

In [168]:
critical_months = {'Establishment': ['March', 'April'], 'Flowering': ['July', 'August']}
critical_months = [m for months in critical_months.values() for m in months]

In [169]:
def critical_features(monthly_avg, colname):
    
    df = monthly_avg[monthly_avg['Month'].isin(critical_months)]
    pivot = df.pivot(index='Year', columns='Month', values='Measurement')
    pivot.columns = [f'{colname}_{m}' for m in pivot.columns]

    return pivot.reset_index()

In [170]:
X_critical = (critical_features(precip_m, 'Precip')
              .merge(critical_features(sun_m, 'Sun'), on='Year')
              .merge(critical_features(tmax_m, 'TempMax'), on='Year')
              .merge(critical_features(tmin_m, 'TempMin'), on='Year'))

X_critical = X_critical.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)
y_critical = X_critical['Yield_Detrended']
X_critical = X_critical.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])

In [171]:
rmse, r2 = run_xgboost(X_critical, y_critical, params_xgboost)

print("Critical Months")
print(f"Features: {X_critical.shape[1]}")
print(f"RMSE: {rmse:.3f}  R2: {r2:.3f}")

Critical Months
Features: 16
RMSE: 0.715  R2: -0.132


### Testing Top Monthly Features

In [172]:
k = 5
y_true, y_pred = [], []

for train_index, test_index in loo.split(X):
    X_train, y_train = X.iloc[train_index], y.iloc[train_index]
    
    correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)
    top_features = correlations.head(k).index.tolist()

    model = xgb.XGBRegressor(random_state=42, **params_xgboost)
    model.fit(X_train[top_features], y_train)

    y_pred.append(model.predict(X.iloc[test_index][top_features])[0])
    y_true.append(y.iloc[test_index].values[0])
 
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"Top {k} Monthly Features")
print(f"RMSE: {rmse:.3f}  R2: {r2:.3f}")


Top 5 Monthly Features
RMSE: 0.751  R2: -0.249


### Gridsearch for best hyperparameters

In [173]:
import itertools

def best_hyperparameters(X_search, y_search, grid):

    parameter_names = list(grid.keys())
    parameter_value_lists = list(grid.values())
    
    best_result = {'r2': -np.inf}
 
    for combination in itertools.product(*parameter_value_lists):
        params = dict(zip(parameter_names, combination))
        rmse, r2 = run_xgboost(X_search, y_search, params)

        if r2 > best_result['r2']:
            best_result = {'params': params, 'rmse': rmse, 'r2': r2}
 
    print(f"Best hyperparameters: {best_result['params']}")
    print(f"RMSE: {best_result['rmse']:.3f}  R2: {best_result['r2']:.3f}")

    return best_result['params']

In [174]:
gridsearch = {
    'max_depth': [1, 2, 3, 4, 5, 10],
    'n_estimators': [30, 40, 50, 60, 70, 80, 90, 100],
    'reg_alpha': [0.5, 1.0, 2.0, 5.0, 10.0],
    'reg_lambda': [0.5, 1.0, 2.0, 5.0, 10.0],
}

print("Grid search")
gridsearch_results = best_hyperparameters(X, y, gridsearch) 

Grid search
Best hyperparameters: {'max_depth': 10, 'n_estimators': 30, 'reg_alpha': 2.0, 'reg_lambda': 10.0}
RMSE: 0.685  R2: -0.039


### Grouping the months into seasons

In [175]:
season_map = {
    'September': 'Autumn', 'October': 'Autumn', 'November': 'Autumn',
    'December': 'Winter', 'January': 'Winter', 'February': 'Winter',
    'March': 'Spring', 'April': 'Spring', 'May': 'Spring',
    'June': 'Summer', 'July': 'Summer', 'August': 'Summer',
}

roll_forward = ['September', 'October', 'November', 'December']

In [176]:
def weather_station(weather_data, colname, method):

    if method == 'mean':
        monthly = weather_data.groupby(['Year', 'Month'])['Measurement'].mean().reset_index()

    elif method == 'median':
        monthly = weather_data.groupby(['Year', 'Month'])['Measurement'].median().reset_index()
        
    else:
        monthly = weather_data[weather_data['Location'] == method][['Year', 'Month', 'Measurement']].reset_index(drop=True)
 
    monthly['Season'] = monthly['Month'].map(season_map)
    rollOn_month = monthly['Month'].isin(roll_forward)

    monthly['HarvestYear'] = np.where(rollOn_month, monthly['Year'] + 1, monthly['Year'])
 
    season_avg = monthly.groupby(['HarvestYear', 'Season'])['Measurement'].mean().reset_index()

    pivot = season_avg.pivot(index='HarvestYear', columns='Season', values='Measurement')
    pivot.columns = [f'{colname}_{s}' for s in pivot.columns]

    return pivot.reset_index().rename(columns={'HarvestYear': 'Year'})
 

In [177]:
def build_features_maxmin(method):
    return (weather_station(precipitation_raw, 'Precip', method)
            .merge(weather_station(sunshine_raw, 'Sun', method), on='Year')
            .merge(weather_station(max_temp_raw, 'TempMax', method), on='Year')
            .merge(weather_station(min_temp_raw, 'TempMin', method), on='Year'))

In [178]:
seasonal_features_maxmin = build_features_maxmin('mean')

print("Seasonal Aggregation")

for csv_path, crop_name in crops_info:
    yield_df_crop = detrend_yield(pd.read_csv(csv_path)[['Year', 'Tonnes']])

    data_seasonal = seasonal_features_maxmin.merge(yield_df_crop, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X_seasonal = data_seasonal.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y_seasonal = data_seasonal['Yield_Detrended']

    rmse, r2 = run_xgboost(X_seasonal, y_seasonal, params_xgboost)
    print(f"  {crop_name:15s} Features: {X_seasonal.shape[1]}  RMSE: {rmse:.3f}  R2: {r2:.3f}")

Seasonal Aggregation
  SpringWheat     Features: 16  RMSE: 0.615  R2: 0.153
  WinterWheat     Features: 16  RMSE: 0.918  R2: -0.087
  SpringBarley    Features: 16  RMSE: 0.424  R2: 0.469
  WinterBarley    Features: 16  RMSE: 0.672  R2: -0.008
  SpringOats      Features: 16  RMSE: 0.409  R2: 0.314
  WinterOats      Features: 16  RMSE: 0.493  R2: -0.154
  Potatoes        Features: 16  RMSE: 3.115  R2: 0.314
  OilseedRape     Features: 16  RMSE: 0.453  R2: -0.060
  BeansAndPeas    Features: 16  RMSE: 0.648  R2: 0.013


In [179]:
def weather_station_naive(weather_data, colname):

    monthly = weather_data.groupby(['Year', 'Month'])['Measurement'].mean().reset_index()
    monthly['Season'] = monthly['Month'].map(season_map)

    season_avg = monthly.groupby(['Year', 'Season'])['Measurement'].mean().reset_index()

    pivot = season_avg.pivot(index='Year', columns='Season', values='Measurement')
    pivot.columns = [f'{colname}_{s}' for s in pivot.columns]
    
    return pivot.reset_index()

In [180]:
def build_features_naive():
    return (weather_station_naive(precipitation_raw, 'Precip')
            .merge(weather_station_naive(sunshine_raw, 'Sun'), on='Year')
            .merge(weather_station_naive(meanTemp, 'TempMean'), on='Year'))

In [181]:
def build_features_meanTemp(method):
    return (weather_station(precipitation_raw, 'Precip', method)
            .merge(weather_station(sunshine_raw, 'Sun', method), on='Year')
            .merge(weather_station(meanTemp, 'TempMean', method), on='Year'))

In [182]:
features_naive = build_features_naive()
features_corrected = build_features_meanTemp('mean')

In [183]:
alignment_results = []

for csv_path, crop_name in crops_info:
    yield_df_crop = detrend_yield(pd.read_csv(csv_path)[['Year', 'Tonnes']])
 
    data_naive = features_naive.merge(yield_df_crop, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X_naive = data_naive.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y_naive = data_naive['Yield_Detrended']
    _, r2_naive = run_xgboost(X_naive, y_naive, params_xgboost)
 
    data_corrected = features_corrected.merge(yield_df_crop, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X_corrected = data_corrected.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y_corrected = data_corrected['Yield_Detrended']
    _, r2_corrected = run_xgboost(X_corrected, y_corrected, params_xgboost)
 
    print(f"  {crop_name:15s} Naive R2={r2_naive:.3f}   Corrected R2={r2_corrected:.3f}")
    alignment_results.append({'crop': crop_name, 'r2_naive': r2_naive, 'r2_corrected': r2_corrected})
 


  SpringWheat     Naive R2=0.278   Corrected R2=0.272
  WinterWheat     Naive R2=0.037   Corrected R2=-0.100
  SpringBarley    Naive R2=0.251   Corrected R2=0.414
  WinterBarley    Naive R2=0.174   Corrected R2=0.008
  SpringOats      Naive R2=0.118   Corrected R2=0.337
  WinterOats      Naive R2=-0.015   Corrected R2=-0.057
  Potatoes        Naive R2=0.155   Corrected R2=0.401
  OilseedRape     Naive R2=-0.454   Corrected R2=-0.343
  BeansAndPeas    Naive R2=0.076   Corrected R2=0.082


### Raw vs Detrended Yield

In [184]:
raw_vs_detrended_results = []

for csv_path, crop_name in crops_info:
    raw_yield = pd.read_csv(csv_path)[['Year', 'Tonnes']]
 
    data_raw = features_corrected.merge(raw_yield, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X_raw = data_raw.drop(columns=['Year', 'Tonnes'])
    y_raw = data_raw['Tonnes']
    _, r2_raw = run_xgboost(X_raw, y_raw, params_xgboost)
 
    detrended_yield = detrend_yield(raw_yield)
    data_det = features_corrected.merge(detrended_yield, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X_det = data_det.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y_det = data_det['Yield_Detrended']
    _, r2_det = run_xgboost(X_det, y_det, params_xgboost)
 
    print(f" {crop_name:15s} Raw R2={r2_raw:.3f}    Detrended R2={r2_det:.3f}")

    raw_vs_detrended_results.append({'crop': crop_name, 'r2_raw': r2_raw, 'r2_detrended': r2_det})
 

 SpringWheat     Raw R2=0.129    Detrended R2=0.272
 WinterWheat     Raw R2=-0.008    Detrended R2=-0.100
 SpringBarley    Raw R2=-0.141    Detrended R2=0.414
 WinterBarley    Raw R2=-0.503    Detrended R2=0.008
 SpringOats      Raw R2=-0.145    Detrended R2=0.337
 WinterOats      Raw R2=0.162    Detrended R2=-0.057
 Potatoes        Raw R2=0.025    Detrended R2=0.401
 OilseedRape     Raw R2=0.007    Detrended R2=-0.343
 BeansAndPeas    Raw R2=-0.293    Detrended R2=0.082


### Weather Station Testing

In [185]:
options = ['mean', 'median', 'Shannon Airport', 'Dublin Airport', 'Cork Airport']

for csv_path, crop_name in crops_info:
    yield_df_crop = detrend_yield(pd.read_csv(csv_path)[['Year', 'Tonnes']])
    print(f"{crop_name}")

    for option in options:
        features_station = build_features_meanTemp(option)
        data_station = features_station.merge(yield_df_crop, on='Year').dropna().sort_values('Year').reset_index(drop=True)

        X_station = data_station.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
        y_station = data_station['Yield_Detrended']

        rmse, r2 = run_xgboost(X_station, y_station, params_xgboost)
        print(f"  {option:20s} RMSE:{rmse:.3f} R2:{r2:.3f}")

SpringWheat
  mean                 RMSE:0.570 R2:0.272
  median               RMSE:0.626 R2:0.120
  Shannon Airport      RMSE:0.674 R2:-0.019
  Dublin Airport       RMSE:0.664 R2:0.011
  Cork Airport         RMSE:0.593 R2:0.211
WinterWheat
  mean                 RMSE:0.924 R2:-0.100
  median               RMSE:0.977 R2:-0.231
  Shannon Airport      RMSE:0.982 R2:-0.243
  Dublin Airport       RMSE:0.877 R2:0.007
  Cork Airport         RMSE:0.832 R2:0.107
SpringBarley
  mean                 RMSE:0.446 R2:0.414
  median               RMSE:0.529 R2:0.175
  Shannon Airport      RMSE:0.530 R2:0.170
  Dublin Airport       RMSE:0.600 R2:-0.061
  Cork Airport         RMSE:0.492 R2:0.285
WinterBarley
  mean                 RMSE:0.667 R2:0.008
  median               RMSE:0.651 R2:0.053
  Shannon Airport      RMSE:0.761 R2:-0.292
  Dublin Airport       RMSE:0.624 R2:0.130
  Cork Airport         RMSE:0.649 R2:0.059
SpringOats
  mean                 RMSE:0.402 R2:0.337
  median               RMSE:0.

### Mean vs Max+Min Temperature

In [186]:
crops_data = [(name, pd.read_csv(path)[['Year', 'Tonnes']]) for path, name in crops_info]
 
comparison_results = []
 
for crop_name, crop in crops_data:
    crop_yield_detrended = detrend_yield(crop)
    print(f"  {crop_name}")
 
    #Mean Temp
    data = features_corrected.merge(crop_yield_detrended, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X_temp_test = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y_temp_test = data['Yield_Detrended']
 
    rmse, r2 = run_xgboost(X_temp_test, y_temp_test, params_xgboost)
    print(f"    {'Mean Temp':15s} RMSE={rmse:.3f}, R2={r2:.3f}")
    comparison_results.append({'crop': crop_name, 'temperature_type': 'Mean Temp', 'r2': r2, 'rmse': rmse})
 
    #Max+Min Temp
    data = seasonal_features_maxmin.merge(crop_yield_detrended, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X_temp_test = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y_temp_test = data['Yield_Detrended']
 
    rmse, r2 = run_xgboost(X_temp_test, y_temp_test, params_xgboost)
    print(f"    {'Max+Min Temp':15s} RMSE={rmse:.3f}, R2={r2:.3f}")
    comparison_results.append({'crop': crop_name, 'temperature_type': 'Max+Min Temp', 'r2': r2, 'rmse': rmse})
 
comparison_table = pd.DataFrame(comparison_results)
 
print(comparison_table.pivot(index='crop', columns='temperature_type', values='r2').round(3))

  SpringWheat
    Mean Temp       RMSE=0.570, R2=0.272
    Max+Min Temp    RMSE=0.615, R2=0.153
  WinterWheat
    Mean Temp       RMSE=0.924, R2=-0.100
    Max+Min Temp    RMSE=0.918, R2=-0.087
  SpringBarley
    Mean Temp       RMSE=0.446, R2=0.414
    Max+Min Temp    RMSE=0.424, R2=0.469
  WinterBarley
    Mean Temp       RMSE=0.667, R2=0.008
    Max+Min Temp    RMSE=0.672, R2=-0.008
  SpringOats
    Mean Temp       RMSE=0.402, R2=0.337
    Max+Min Temp    RMSE=0.409, R2=0.314
  WinterOats
    Mean Temp       RMSE=0.472, R2=-0.057
    Max+Min Temp    RMSE=0.493, R2=-0.154
  Potatoes
    Mean Temp       RMSE=2.909, R2=0.401
    Max+Min Temp    RMSE=3.115, R2=0.314
  OilseedRape
    Mean Temp       RMSE=0.510, R2=-0.343
    Max+Min Temp    RMSE=0.453, R2=-0.060
  BeansAndPeas
    Mean Temp       RMSE=0.625, R2=0.082
    Max+Min Temp    RMSE=0.648, R2=0.013
temperature_type  Max+Min Temp  Mean Temp
crop                                     
BeansAndPeas             0.013      0.082
Oilse

### Linear Model Testing

In [187]:
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler

ridge_lasso_results = []

for csv_path, crop_name in crops_info:
 
    yield_df_crop = detrend_yield(pd.read_csv(csv_path)[['Year', 'Tonnes']])
    data = features_corrected.merge(yield_df_crop, on='Year').dropna().sort_values('Year').reset_index(drop=True)
 
    X_crop = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y_crop = data['Yield_Detrended']
 
    crop_result = {'crop': crop_name}
 
    #Ridge
    y_true, y_pred = [], []
 
    for train_index, test_index in loo.split(X_crop):
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_crop.iloc[train_index])
        X_test_scaled = scaler.transform(X_crop.iloc[test_index])
 
        model = RidgeCV()
        model.fit(X_train_scaled, y_crop.iloc[train_index])
 
        y_pred.append(model.predict(X_test_scaled)[0])
        y_true.append(y_crop.iloc[test_index].values[0])
 
    crop_result['r2_ridge'] = r2_score(y_true, y_pred)
 
    #Lasso
    y_true, y_pred = [], []
 
    for train_index, test_index in loo.split(X_crop):
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_crop.iloc[train_index])
        X_test_scaled = scaler.transform(X_crop.iloc[test_index])
 
        model = LassoCV(max_iter=5000)
        model.fit(X_train_scaled, y_crop.iloc[train_index])
 
        y_pred.append(model.predict(X_test_scaled)[0])
        y_true.append(y_crop.iloc[test_index].values[0])
 
    crop_result['r2_lasso'] = r2_score(y_true, y_pred)
 
    print(f"  {crop_name:15s} Ridge R2={crop_result['r2_ridge']:.3f}   Lasso R2={crop_result['r2_lasso']:.3f}")
 
    ridge_lasso_results.append(crop_result)
 


  SpringWheat     Ridge R2=-0.164   Lasso R2=-0.161
  WinterWheat     Ridge R2=0.017   Lasso R2=-0.266
  SpringBarley    Ridge R2=0.056   Lasso R2=-0.075
  WinterBarley    Ridge R2=-0.169   Lasso R2=-0.153
  SpringOats      Ridge R2=0.008   Lasso R2=-0.086
  WinterOats      Ridge R2=-0.075   Lasso R2=-0.343
  Potatoes        Ridge R2=-0.168   Lasso R2=-0.084
  OilseedRape     Ridge R2=-0.256   Lasso R2=-0.058
  BeansAndPeas    Ridge R2=-0.327   Lasso R2=-0.125


### Monthly Window Testing

In [188]:
growth_year = ['September', 'October', 'November', 'December', 'January', 'February',
               'March', 'April', 'May', 'June', 'July', 'August']

window_results = []
 
for start_position in range(12):
    window = [growth_year[(start_position + offset) % 12] for offset in range(4)]
 
    def build_window(monthly_avg, colname):
        df = monthly_avg[monthly_avg['Month'].isin(window)].copy()

        rollOn_month = df['Month'].isin(roll_forward)
        df['HarvestYear'] = np.where(rollOn_month, df['Year'] + 1, df['Year'])
        
        window_avg = df.groupby('HarvestYear')['Measurement'].mean().reset_index()

        return window_avg.rename(columns={'Measurement': colname, 'HarvestYear': 'Year'})
 
    window_features = (build_window(precip_m, 'Precip')
                        .merge(build_window(sun_m, 'Sun'), on='Year')
                        .merge(build_window(tmax_m, 'TempMax'), on='Year')
                        .merge(build_window(tmin_m, 'TempMin'), on='Year'))
 
    data_window = window_features.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X_window = data_window.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y_window = data_window['Yield_Detrended']
 
    rmse, r2 = run_xgboost(X_window, y_window, params_xgboost)
    window_label = f"{window[0]}-{window[-1]}"
    window_results.append({'window': window_label, 'rmse': rmse, 'r2': r2})
 
window_df = pd.DataFrame(window_results).sort_values('r2', ascending=False)
print(window_df.to_string(index=False))
 

            window     rmse        r2
        May-August 0.528557  0.381497
        April-July 0.581873  0.250427
    June-September 0.611342  0.172581
      July-October 0.626244  0.131750
   August-November 0.659033  0.038449
   October-January 0.676268 -0.012500
        March-June 0.694498 -0.067823
    December-March 0.722174 -0.154625
September-December 0.744459 -0.242217
      February-May 0.758855 -0.274896
     January-April 0.768107 -0.306174
 November-February 0.792905 -0.391875


### Permutation Test Best Window Found

In [189]:
def build_window_final(monthly_avg, colname):

    df = monthly_avg[monthly_avg['Month'].isin(window)].copy()
    window_avg = df.groupby('Year')['Measurement'].mean().reset_index()
    
    return window_avg.rename(columns={'Measurement': colname})

In [190]:
window = ['May', 'June', 'July', 'August']
 
window_features = (build_window_final(precip_m, 'Precip')
                    .merge(build_window_final(sun_m, 'Sun'), on='Year')
                    .merge(build_window_final(tmax_m, 'TempMax'), on='Year')
                    .merge(build_window_final(tmin_m, 'TempMin'), on='Year'))

In [191]:
data = window_features.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)
X_window = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
y_window = data['Yield_Detrended']
 
_, r2 = run_xgboost(X_window, y_window, params_xgboost)
 
shuffles = 2000
random_number = np.random.default_rng(42)
r2_scores = []

for shuffle_number in range(shuffles):
    shuffled_y_series = pd.Series(random_number.permutation(y_window.values), index=y_window.index)
    _, shuffled_r2 = run_xgboost(X_window, shuffled_y_series, params_xgboost)
    r2_scores.append(shuffled_r2)
 
r2_scores = np.array(r2_scores)
p_value = (np.sum(r2_scores >= r2) + 1) / (shuffles + 1)
 
print(f"\nR2 (May-August): {r2:.3f}")
print(f"p-value: {p_value:.4f}")


R2 (May-August): 0.381
p-value: 0.0005


### Min and Max for RMSE Comparison

In [192]:
significant_crops = [
    (folderPath + r"\CropYield\SpringWheat1985_2025.csv", 'SpringWheat'),
    (folderPath + r"\CropYield\SpringBarley1985_2025.csv", 'SpringBarley'),
    (folderPath + r"\CropYield\SpringOats1985_2025.csv", 'SpringOats'),
    (folderPath + r"\CropYield\Potatoes1985_2025.csv", 'Potatoes'),
]
 
print(f"{'Crop':15s} {'Min (t/ha)':>12s} {'Max (t/ha)':>12s} {'RMSE (t/ha)':>13s}")
 
for csv_path, crop_name in significant_crops:
    raw_yield = pd.read_csv(csv_path)[['Year', 'Tonnes']]
    yield_min = raw_yield['Tonnes'].min()
    yield_max = raw_yield['Tonnes'].max()
 
    yield_df_crop = detrend_yield(raw_yield)
    data = features_corrected.merge(yield_df_crop, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y = data['Yield_Detrended']
 
    rmse, r2 = run_xgboost(X, y, params_xgboost)
 
    print(f"{crop_name:15s} {yield_min:12.2f} {yield_max:12.2f} {rmse:13.3f}")

Crop              Min (t/ha)   Max (t/ha)   RMSE (t/ha)
SpringWheat             5.40         8.70         0.570
SpringBarley            4.80         8.10         0.446
SpringOats              4.40         7.90         0.402
Potatoes               20.30        45.60         2.909
